# Prism Risk Engine: Core Portfolio & Risk Metrics (Phase 2 Deliverable)

This notebook demonstrates the calculation and visual validation of **Core Metrics (FR-M1 to M6)** on the calibrated multi-asset market data snapshot.

- **Portfolio**: 60% Equity (`NIFTYBEES.NS`), 20% Gold (`GOLDBEES.NS`), 20% Debt (`LIQUIDBEES.NS`)
- **Benchmark**: `^NSEI` (Nifty 50 Index)
- **Risk-Free Rate**: 6.5% Annual (Compounded Daily)
- **Trading Days**: 252 days/year
- **Sign Convention**: Drawdowns & negative returns are strictly $\le 0$.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from prismrisk.config import Config
from prismrisk.data.providers import SnapshotProvider
from prismrisk.data.calendar import align_prices
from prismrisk.metrics import (
    simple_returns,
    portfolio_returns,
    cumulative_returns,
    cagr,
    annualised_vol,
    rolling_vol,
    drawdown_series,
    max_drawdown,
    calmar_ratio,
    sharpe_ratio,
    sortino_ratio,
    beta,
    tracking_error,
    information_ratio,
    correlation_matrix,
)

print("Prism Risk Engine modules loaded successfully.")

## 1. Ingest Data & Align Common Sample Window

In [ ]:
cfg = Config.load(project_root / "configs/demo.yaml")
snapshot_path = project_root / "data/snapshot/market_data_snapshot.parquet"

provider = SnapshotProvider(snapshot_path)
tickers = {name: asset.ticker for name, asset in cfg.assets.items()}
raw_prices = provider.fetch_all(tickers=tickers, start=cfg.data.start, end=cfg.data.end)

aligned_prices, start_dates = align_prices(raw_prices, max_ffill_days=cfg.data.max_ffill_days)
rets = simple_returns(aligned_prices)

weights = pd.Series(cfg.portfolio.weights)
p_rets = portfolio_returns(rets[list(weights.index)], weights)
bm_rets = rets["benchmark"]

print(f"Effective Sample Period: {aligned_prices.index[0].date()} to {aligned_prices.index[-1].date()}")
print(f"Total Trading Days: {len(aligned_prices)}")

## 2. Core Metrics Summary Table

Comparing the **60/20/20 Multi-Asset Portfolio** against the **Benchmark**.

In [ ]:
rf = cfg.risk_free.annual_rate
p_dd = max_drawdown(p_rets)
bm_dd = max_drawdown(bm_rets)

metrics_data = {
    "Metric": [
        "CAGR",
        "Annualised Volatility",
        "Sharpe Ratio (rf=6.5%)",
        "Sortino Ratio",
        "Max Drawdown (MDD)",
        "Calmar Ratio",
        "Beta vs Benchmark",
        "Tracking Error",
        "Information Ratio",
    ],
    "Portfolio (60/20/20)": [
        f"{cagr(p_rets):.2%}",
        f"{annualised_vol(p_rets):.2%}",
        f"{sharpe_ratio(p_rets, rf_annual=rf):.2f}",
        f"{sortino_ratio(p_rets, rf_annual=rf):.2f}",
        f"{p_dd.depth:.2%}",
        f"{calmar_ratio(p_rets):.2f}",
        f"{beta(p_rets, bm_rets):.2f}",
        f"{tracking_error(p_rets, bm_rets):.2%}",
        f"{information_ratio(p_rets, bm_rets):.2f}",
    ],
    "Benchmark (^NSEI)": [
        f"{cagr(bm_rets):.2%}",
        f"{annualised_vol(bm_rets):.2%}",
        f"{sharpe_ratio(bm_rets, rf_annual=rf):.2f}",
        f"{sortino_ratio(bm_rets, rf_annual=rf):.2f}",
        f"{bm_dd.depth:.2%}",
        f"{calmar_ratio(bm_rets):.2f}",
        "1.00",
        "0.00%",
        "0.00",
    ]
}

metrics_df = pd.DataFrame(metrics_data).set_index("Metric")
metrics_df

## 3. Visual Charts (Cumulative Return, Drawdown, Rolling Volatility)

In [ ]:
# Palette constants from design spec
COLOR_PORTFOLIO = "#111827"
COLOR_BENCHMARK = "#6c757d"
COLOR_DRAWDOWN = "#c0392b"

fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)

# Chart 1: Cumulative Return
cum_p = cumulative_returns(p_rets)
cum_bm = cumulative_returns(bm_rets)
axes[0].plot(cum_p.index, cum_p * 100, label="Portfolio (60/20/20)", color=COLOR_PORTFOLIO, linewidth=2)
axes[0].plot(cum_bm.index, cum_bm * 100, label="Benchmark (^NSEI)", color=COLOR_BENCHMARK, linewidth=1.5, linestyle="--")
axes[0].set_title("Cumulative Return vs Benchmark", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Cumulative Return (%)")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# Chart 2: Underwater Drawdown Area Chart
dd_series_p = drawdown_series(p_rets)
axes[1].fill_between(dd_series_p.index, dd_series_p * 100, 0, color=COLOR_DRAWDOWN, alpha=0.3)
axes[1].plot(dd_series_p.index, dd_series_p * 100, color=COLOR_DRAWDOWN, linewidth=1.5)
axes[1].scatter([p_dd.trough_date], [p_dd.depth * 100], color="#900C3F", s=60, zorder=5, label=f"Max DD: {p_dd.depth:.1%}")
axes[1].set_title("Portfolio Underwater Drawdown Series", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Drawdown (%)")
axes[1].legend(loc="lower left")
axes[1].grid(True, alpha=0.3)

# Chart 3: Rolling Volatility (30d and 90d)
roll_30 = rolling_vol(p_rets, window=30)
roll_90 = rolling_vol(p_rets, window=90)
axes[2].plot(roll_30.index, roll_30 * 100, label="Rolling Vol 30d", color="#1f77b4", linewidth=1.2)
axes[2].plot(roll_90.index, roll_90 * 100, label="Rolling Vol 90d", color="#d4a017", linewidth=1.8)
axes[2].set_title("Portfolio Rolling Volatility (30 vs 90 Trading Days)", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Annualised Vol (%)")
axes[2].set_xlabel("Date")
axes[2].legend(loc="upper right")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / "docs/core_metrics_visuals.png", dpi=150)
plt.show()
print("Visual charts generated and saved to docs/core_metrics_visuals.png")

## 4. Multi-Asset Correlation Heatmap

In [ ]:
corr = correlation_matrix(rets)
print("Pairwise Correlation Matrix:")
corr.round(3)